In [ ]:
# import pandas as pd

In [ ]:
# df =pd.read_csv(r"C:\Users\Homes\Downloads\sodapdf-converted.csv")
# df.head(10)

,lines


In [ ]:
# import requests
# from bs4 import BeautifulSoup
# import pandas as pd

In [4]:


# import requests

# def scrape_airline_reviews(url):
#     """Scrapes airline reviews from a single page."""

#     headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36'}  #  **Replace with your actual User-Agent**
#     try:
#         response = requests.get(url, headers=headers)
#         response.raise_for_status()  #  Check for HTTP errors
#     except requests.exceptions.RequestException as e:
#         print(f"Error fetching {url}: {e}")
#         return []

#     soup = BeautifulSoup(response.content, 'html.parser')
#     reviews_data = []

#     # review_containers = soup.find_all('article', itemprop='review')  #  Find each review container
#     # review_containers = soup.find_all('div', class_='review-card')  #  Adjusted to match the new structure
#     review_containers = soup.find_all('table', class_='review-ratings')

#     for container in review_containers:
#         try:
#             #  Initialize an empty dictionary for each review
#             review = {
#                 "Aircraft": None,
#                 "Type Of Traveller": None,
#                 "Seat Type": None,
#                 "Route": None,
#                 "Date Flown": None,
#                 "Seat Comfort": None,
#                 "Cabin Staff Service": None,
#                 "Ground Service": None,
#                 "Value For Money": None,
#                 "Recommended": None,
#                 # "Review Text": None  #  To capture the main review text
#             }

#             #  Extract the review text
#             # text_element = container.find('div', class_='text_content')
#             # review["Review Text"] = text_element.text.strip() if text_element else None

#             #  Extract the details from the review
#             detail_items = container.find_all('h3', class_='text10px')
#             for item in detail_items:
#                 text = item.text.strip()
#                 if "Aircraft" in text:
#                     review["Aircraft"] = text.replace("Aircraft", "").strip()
#                 elif "Type Of Traveller" in text:
#                     review["Type Of Traveller"] = text.replace("Type Of Traveller", "").strip()
#                 elif "Seat Type" in text:
#                     review["Seat Type"] = text.replace("Seat Type", "").strip()
#                 elif "Route" in text:
#                     review["Route"] = text.replace("Route", "").strip()
#                 elif "Date Flown" in text:
#                     review["Date Flown"] = text.replace("Date Flown", "").strip()
#                 elif "Seat Comfort" in text:
#                     review["Seat Comfort"] = text.replace("Seat Comfort", "").strip()
#                 elif "Cabin Staff Service" in text:
#                     review["Cabin Staff Service"] = text.replace("Cabin Staff Service", "").strip()
#                 elif "Ground Service" in text:
#                     review["Ground Service"] = text.replace("Ground Service", "").strip()
#                 elif "Value For Money" in text:
#                     review["Value For Money"] = text.replace("Value For Money", "").strip()
#                 elif "Recommended" in text:
#                     review["Recommended"] = text.replace("Recommended", "").strip()

#             reviews_data.append(review)

#         except Exception as e:
#             print(f"Error processing a review: {e}")

#     return reviews_data

# def scrape_all_pages(base_url, num_pages=10):  #  Adjust num_pages as needed
#     """Scrapes reviews from multiple pages."""

#     all_reviews = []
#     for page_num in range(1, num_pages + 1):
#         url = f"{base_url}?sortby=post_date%3ADesc&pagesize=100&page={page_num}"  #  Construct the URL for each page
#         print(f"Scraping: {url}")  #  To see which page is being scraped
#         all_reviews.extend(scrape_airline_reviews(url))
#         import time
#         time.sleep(2)  #  Be respectful to the server
#     return all_reviews

# #  ---  Main Execution  ---
# base_url = "https://www.airlinequality.com/airline-reviews/british-airways/"
# all_reviews = scrape_all_pages(base_url, num_pages=5)  #  Scrape the first 5 pages

# if all_reviews:
#     df = pd.DataFrame(all_reviews)
#     print(df)
#     df.to_csv("british_airways_reviews.csv", index=False, encoding="utf-8")  #  Save to CSV
#     #  df.to_excel("british_airways_reviews.xlsx", index=False, engine='openpyxl')  #  Save to Excel (if you have openpyxl installed)
# else:
#     print("No reviews were scraped.")

In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_airline_reviews(url):
    """Scrapes airline reviews from a single page."""

    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36'}  #  **Replace with your actual User-Agent**
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()  #  Check for HTTP errors
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return []

    soup = BeautifulSoup(response.content, 'html.parser')
    reviews_data = []

    review_containers = soup.find_all('article', itemprop='review')  #  Find each review container

    for container in review_containers:
        try:
            review = {
                "Aircraft": None,
                "Type Of Traveller": None,
                "Seat Type": None,
                "Route": None,
                "Date Flown": None,
                "Seat Comfort": None,
                "Cabin Staff Service": None,
                "Ground Service": None,
                "Value For Money": None,
                "Recommended": None,
                "Review Text": None  #  To capture the main review text
            }

            #  Extract the review text
            text_element = container.find('div', class_='text_content')
            review["Review Text"] = text_element.text.strip() if text_element else None

            #  ---  NEW CODE TO EXTRACT FROM THE TABLE  ---
            table = container.find('table', class_='review-ratings')  # Find the table
            if table:  # Only proceed if the table is found
                for row in table.find_all('tr'):  # Iterate through each row in the table
                    header_cell = row.find('td', class_='review-rating-header')
                    value_cell = row.find('td', class_='review-value')
                    if header_cell and value_cell:  # Only process if both cells are found
                        header = header_cell.text.strip()
                        value = value_cell.text.strip()
                        if header in review:  # Check if the header is a key we want
                            review[header] = value

                    # Extract star ratings
                    if header_cell:
                        header = header_cell.text.strip()
                        if "Comfort" in header or "Service" in header or "Money" in header or "Ground" in header:
                            star_cells = row.find_all('span', class_='star fill')
                            rating = len(star_cells)
                            review[header] = rating

            reviews_data.append(review)

        except Exception as e:
            print(f"Error processing a review: {e}")

    return reviews_data

def scrape_all_pages(base_url, num_pages=20):  #  Adjust num_pages as needed
    """Scrapes reviews from multiple pages."""

    all_reviews = []
    for page_num in range(1, num_pages + 1):
        url = f"{base_url}?sortby=post_date%3ADesc&pagesize=100&page={page_num}"  #  Construct the URL for each page
        print(f"Scraping: {url}")  #  To see which page is being scraped
        all_reviews.extend(scrape_airline_reviews(url))
        import time
        time.sleep(2)  #  Be respectful to the server
    return all_reviews

#  ---  Main Execution  ---
base_url = "https://www.airlinequality.com/airline-reviews/british-airways/"
all_reviews = scrape_all_pages(base_url, num_pages=5)  #  Scrape the first 5 pages

if all_reviews:
    df = pd.DataFrame(all_reviews)
    print(df)
    df.to_csv("british_airways_reviews.csv", index=False, encoding="utf-8")
    #  df.to_excel("british_airways_reviews.xlsx", index=False, engine='openpyxl')
else:
    print("No reviews were scraped.")

Scraping: https://www.airlinequality.com/airline-reviews/british-airways/?sortby=post_date%3ADesc&pagesize=100&page=1
Scraping: https://www.airlinequality.com/airline-reviews/british-airways/?sortby=post_date%3ADesc&pagesize=100&page=2
Scraping: https://www.airlinequality.com/airline-reviews/british-airways/?sortby=post_date%3ADesc&pagesize=100&page=3
Scraping: https://www.airlinequality.com/airline-reviews/british-airways/?sortby=post_date%3ADesc&pagesize=100&page=4
Scraping: https://www.airlinequality.com/airline-reviews/british-airways/?sortby=post_date%3ADesc&pagesize=100&page=5
           Aircraft Type Of Traveller       Seat Type  \
0              A380      Solo Leisure   Economy Class   
1              A319          Business  Business Class   
2          A321 Neo    Family Leisure   Economy Class   
3        Boeing 777      Solo Leisure  Business Class   
4              None    Family Leisure   Economy Class   
..              ...               ...             ...   
495        

In [7]:
import os

# Ensure the directory exists
os.makedirs("BritisAirline", exist_ok=True)

# Save the DataFrame to a CSV file
df.to_csv("BritisAirline/BA_reviews.csv")